In [2]:
import torch
import platform

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Total GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / (1024 ** 3),
            2
        ),
        "GB"
    )
else:
    print("Running on CPU")

Python: 3.11.9
PyTorch: 2.14.1+cu126
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
Total GPU memory: 4.0 GB


In [3]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

PyTorch version: 2.14.1+cu126
CUDA build: 12.6
CUDA available: True
GPU count: 1


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModelForSeq2SeqLM, Trainer, TrainingArguments, Seq2SeqTrainer, Seq2SeqTrainingArguments
from datasets import load_dataset

In [3]:
import os
import re
import gc
import random
import unicodedata
import regex
import numpy as np
import torch

from pathlib import Path
from collections import Counter

from datasets import (
    load_dataset,
    load_from_disk,
    Dataset,
    DatasetDict,
    Features,
    Sequence,
    Value
)

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import classification_report, confusion_matrix

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODEL_DIR = PROJECT_DIR / "models"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)
print("Model directory:", MODEL_DIR)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

Project directory: e:\punctuation_project
Data directory: e:\punctuation_project\data
Model directory: e:\punctuation_project\models
PyTorch: 2.14.1+cu126
CUDA: True


In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Selected device:", device)

Selected device: cuda


In [3]:
MODEL_NAME = "ai4bharat/indic-bert"

indic_bert_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False
)

print("IndicBERT tokenizer loaded successfully.")
print("Tokenizer class:", type(indic_bert_tokenizer).__name__)

IndicBERT tokenizer loaded successfully.
Tokenizer class: AlbertTokenizer


In [8]:
dataset = load_dataset("cfilt/iitb-english-hindi", split="train")
print(dataset)

README.md:   0%|          | 0.00/3.14k [00:00<?, ?B/s]

d:\punctuation_project\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HOME\.cache\huggingface\hub\datasets--cfilt--iitb-english-hindi. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


dataset_infos.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

d:\punctuation_project\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HOME\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  190MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (4 outputs left out)

Generating train split:   0%|          | 0/1659083 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/520 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2507 [00:00<?, ? examples/s]

Dataset({
    features: ['translation'],
    num_rows: 1659083
})


In [9]:
print("Total examples:", len(dataset))

print("\nFirst example:")
print("Hindi:", dataset[0]["translation"]["hi"])
print("English:", dataset[0]["translation"]["en"])

Total examples: 1659083

First example:
Hindi: अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें
English: Give your application an accessibility workout


In [4]:
import gc

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Imports and memory cleanup completed.")


Imports and memory cleanup completed.


In [5]:
from datasets import load_from_disk

punctuation_path = PROCESSED_DIR / "punctuation_processed"

if not punctuation_path.exists():
    raise FileNotFoundError(
        f"Processed punctuation dataset not found: {punctuation_path}"
    )

punctuation_data = load_from_disk(str(punctuation_path))

print(punctuation_data)
print("Punctuation dataset loaded successfully.")

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2251236
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 278086
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 283654
    })
})
Punctuation dataset loaded successfully.


In [4]:
wiki_dataset = load_dataset(
    "wikimedia/wikipedia",
    "20231101.hi",
    split="train"
)

print("Total articles:", len(wiki_dataset))
print("Columns:", wiki_dataset.column_names)
print("\nSample article:\n")
print(wiki_dataset[0]["text"][:1500])

Total articles: 163093
Columns: ['id', 'url', 'title', 'text']

Sample article:

हम होंगे कामयाब ( का गिरिजा कुमार माथुर द्वारा किया गया हिंदी भावानुवाद) एक प्रतिरोध गीत है। यह गीत बीसवीं सदी में नागरिक अधिकार आंदोलन का प्रधान स्वर बना। इस गीत को आमतौर पर "I'll Overcome Some Day" ("आई विल ओवरकम सम डे") से काव्यावतरित माना जाता है, जो चार्ल्स अल्बर्ट टिंडले द्वारा गाया गया था और जिसे 1900 में पहली बार प्रकाशित किया गया था।

सन्दर्भ
Hum Honge Kamyab Lyrics 
नागरिक अधिकार आंदोलन
देशभक्ति के गीत
आधार


In [6]:
import torch

print("PyTorch version:", torch.__version__)

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch version: 2.14.1+cu126
GPU available: True
GPU name: NVIDIA GeForce RTX 3050 Laptop GPU


In [9]:
def clean_hindi(text):
    text = unicodedata.normalize("NFC", str(text))

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # Remove zero-width space and BOM only
    # Preserve ZWJ and ZWNJ
    text = re.sub(r"[\u200b\ufeff]", "", text)

    # Normalize spaces but preserve paragraph breaks
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


# Split the Hugging Face Dataset directly.
# We split by article so sentences from one article
# cannot accidentally appear in different splits.

wiki_temp = wiki_dataset.train_test_split(
    test_size=0.20,
    seed=42
)

wiki_remaining = wiki_temp["test"].train_test_split(
    test_size=0.50,
    seed=42
)

wiki_splits = DatasetDict({
    "train": wiki_temp["train"],
    "validation": wiki_remaining["train"],
    "test": wiki_remaining["test"]
})

print(wiki_splits)

DatasetDict({
    train: Dataset({
        features: ['id', 'url', 'title', 'text'],
        num_rows: 130474
    })
    validation: Dataset({
        features: ['id', 'url', 'title', 'text'],
        num_rows: 16309
    })
    test: Dataset({
        features: ['id', 'url', 'title', 'text'],
        num_rows: 16310
    })
})


In [10]:
def split_sentences(text):
    sentences = re.split(
        r"(?<=[।॥!?])\s+|(?<=\.)\s+",
        text
    )

    return [
        sentence.strip()
        for sentence in sentences
        if len(sentence.strip().split()) >= 3
    ]


LABEL_MAP = {
    ",": 1,
    ";": 1,
    ":": 1,
    ".": 2,
    "।": 2,
    "॥": 2,
    "?": 3,
    "!": 4
}


def create_punctuation_example(sentence):

    # Unicode-aware word and punctuation extraction
    tokens = regex.findall(
        r"[\p{L}\p{M}\p{N}]+|[।॥.,?!;:]",
        sentence
    )

    words = []
    labels = []

    i = 0

    while i < len(tokens):

        if regex.fullmatch(r"[\p{L}\p{M}\p{N}]+", tokens[i]):

            words.append(tokens[i])

            label = 0
            j = i + 1

            # Find punctuation following this word
            while j < len(tokens) and tokens[j] in LABEL_MAP:
                label = LABEL_MAP[tokens[j]]
                j += 1

            labels.append(label)
            i = j

        else:
            i += 1

    if len(words) < 3:
        return None

    return {
        "words": words,
        "word_labels": labels
    }

In [11]:

import random

sample_articles = wiki_splits["train"].shuffle(
    seed=42
).select(
    range(min(1000, len(wiki_splits["train"])))
)

total_paragraphs = 0
total_sentences = 0
total_groups = 0
short_paragraphs = 0
long_sentences = 0
missing_endings = 0
groups_with_english = 0

for article in sample_articles:

    cleaned_article = clean_hindi(article["text"])

    for paragraph in cleaned_article.split("\n"):

        paragraph = paragraph.strip()

        if not paragraph:
            continue

        total_paragraphs += 1
        sentences = split_sentences(paragraph)

        if len(sentences) < 5:
            short_paragraphs += 1
            continue

        total_sentences += len(sentences)

        for start in range(0, len(sentences) - 4, 5):

            group = sentences[start:start + 5]
            total_groups += 1

            if any(len(s.split()) > 100 for s in group):
                long_sentences += 1

            if any(
                not s.rstrip().endswith(("।", "॥", "!", "?", "."))
                for s in group
            ):
                missing_endings += 1

            if any(
                any("a" <= ch.lower() <= "z" for ch in s)
                for s in group
            ):
                groups_with_english += 1

print("DIAGNOSTIC RESULTS")
print("------------------------------")
print("Articles sampled:", len(sample_articles))
print("Paragraphs:", total_paragraphs)
print("Paragraphs with fewer than 5 sentences:", short_paragraphs)
print("Sentences:", total_sentences)
print("Groups of 5:", total_groups)
print("Groups with unusually long sentences:", long_sentences)
print("Groups with missing sentence endings:", missing_endings)
print("Groups containing English text:", groups_with_english)


DIAGNOSTIC RESULTS
------------------------------
Articles sampled: 1000
Paragraphs: 13394
Paragraphs with fewer than 5 sentences: 12681
Sentences: 5120
Groups of 5: 850
Groups with unusually long sentences: 2
Groups with missing sentence endings: 25
Groups containing English text: 148


In [12]:

from datasets import load_from_disk
from pathlib import Path

ordering_path = (
    Path(r"E:\punctuation_project")
    / "data"
    / "processed"
    / "ordering_processed"
)

ordering_data = load_from_disk(str(ordering_path))

MAX_LENGTH = 512

for split_name in ["train", "validation", "test"]:

    dataset = ordering_data[split_name]

    total = len(dataset)
    truncated = 0
    lengths = []

    for example in dataset:
        length = len(example["input_ids"])
        lengths.append(length)

        if length >= MAX_LENGTH:
            truncated += 1

    print(f"\n{split_name.upper()}")
    print("-" * 35)
    print("Total examples:", total)
    print("Examples reaching 512 tokens:", truncated)
    print(
        "Percentage:",
        round((truncated / total) * 100, 2),
        "%"
    )
    print("Average input length:", round(sum(lengths) / total, 2))
    print("Maximum input length:", max(lengths))



TRAIN
-----------------------------------
Total examples: 121408
Examples reaching 512 tokens: 386
Percentage: 0.32 %
Average input length: 223.73
Maximum input length: 512

VALIDATION
-----------------------------------
Total examples: 14845
Examples reaching 512 tokens: 47
Percentage: 0.32 %
Average input length: 224.65
Maximum input length: 512

TEST
-----------------------------------
Total examples: 15431
Examples reaching 512 tokens: 49
Percentage: 0.32 %
Average input length: 223.97
Maximum input length: 512


In [15]:
punctuation_features = Features({
    "words": Sequence(Value("string")),
    "word_labels": Sequence(Value("int64"))
})


def generate_punctuation_examples(source_dataset):

    for article in source_dataset:

        cleaned_article = clean_hindi(article["text"])

        # Process paragraphs separately
        paragraphs = cleaned_article.split("\n")

        for paragraph in paragraphs:

            if not paragraph.strip():
                continue

            sentences = split_sentences(paragraph)

            for sentence in sentences:

                example = create_punctuation_example(sentence)

                if example is not None:
                    yield example

In [16]:
def build_punctuation_dataset(source_dataset, split_name):

    return Dataset.from_generator(
        generate_punctuation_examples,
        gen_kwargs={
            "source_dataset": source_dataset
        },
        features=punctuation_features,
        split=split_name,
        keep_in_memory=False,
        writer_batch_size=500
    )


punct_train = build_punctuation_dataset(
    wiki_splits["train"], "train"
)

punct_val = build_punctuation_dataset(
    wiki_splits["validation"], "validation"
)

punct_test = build_punctuation_dataset(
    wiki_splits["test"], "test"
)

print("Training examples:", len(punct_train))
print("Validation examples:", len(punct_val))
print("Testing examples:", len(punct_test))

print("\nSample:")
print(punct_train[0])

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Training examples: 2251236
Validation examples: 278086
Testing examples: 283654

Sample:
{'words': ['स्त्री', 'हृदयम', 'मलयालम', 'സ്ത്രീഹൃദയം', 'भारत', 'से', 'मलयालम', 'भाषा', 'की', 'सर्वाधिक', 'प्रतिभावान', 'कवयित्रियों', 'में', 'से', 'एक', 'नालापत', 'बालमणि', 'अम्मा', 'का', 'काव्य', 'संग्रह', 'है', 'जो', '1939', 'में', 'पहली', 'बार', 'मलयालम', 'भाषा', 'में', 'प्रकाशित', 'हुआ'], 'word_labels': [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2]}


In [ ]:
def encode_punctuation_batch(examples):

    batch_input_ids = []
    batch_attention_masks = []
    batch_labels = []

    for words, word_labels in zip(
        examples["words"],
        examples["word_labels"]
    ):

        all_tokens = []
        all_labels = []

        for word, label in zip(words, word_labels):

            pieces = indic_bert_tokenizer.tokenize(word)

            if not pieces:
                continue

            all_tokens.extend(pieces)

            # Assign punctuation label to final subword
            all_labels.extend(
                [-100] * (len(pieces) - 1) + [label]
            )

        # Reserve two positions for CLS and SEP
        MAX_LENGTH = 64

        all_tokens = all_tokens[:MAX_LENGTH - 2]
        all_labels = all_labels[:MAX_LENGTH - 2]

        tokens = (
            [indic_bert_tokenizer.cls_token]
            + all_tokens
            + [indic_bert_tokenizer.sep_token]
        )

        labels = [-100] + all_labels + [-100]

        input_ids = indic_bert_tokenizer.convert_tokens_to_ids(
            tokens
        )

        attention_mask = [1] * len(input_ids)

        batch_input_ids.append(input_ids)
        batch_attention_masks.append(attention_mask)
        batch_labels.append(labels)

    return {
        "input_ids": batch_input_ids,
        "attention_mask": batch_attention_masks,
        "labels": batch_labels
    }


def tokenize_punctuation_data(dataset):

    return dataset.map(
        encode_punctuation_batch,
        batched=True,
        batch_size=32,
        writer_batch_size=100,
        remove_columns=dataset.column_names,
        keep_in_memory=False
    )


punctuation_data = DatasetDict({
    "train": tokenize_punctuation_data(punct_train),
    "validation": tokenize_punctuation_data(punct_val),
    "test": tokenize_punctuation_data(punct_test)
})

print(punctuation_data)

Map:   0%|          | 0/2251236 [00:00<?, ? examples/s]

Map:   0%|          | 0/278086 [00:00<?, ? examples/s]

Map:   0%|          | 0/283654 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2251236
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 278086
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 283654
    })
})


In [18]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

punctuation_save_path = PROCESSED_DIR / "punctuation_processed"

punctuation_data.save_to_disk(str(punctuation_save_path))

print("Dataset saved at:", punctuation_save_path)

Saving the dataset (0/2 shards):   0%|          | 0/2251236 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/278086 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/283654 [00:00<?, ? examples/s]

Dataset saved at: d:\punctuation_project\data\processed\punctuation_processed


In [19]:
import random

def generate_ordering_examples(source_dataset):

    rng = random.Random(42)

    for article in source_dataset:

        cleaned_article = clean_hindi(article["text"])

        # Process each paragraph separately
        paragraphs = cleaned_article.split("\n")

        for paragraph in paragraphs:

            sentences = split_sentences(paragraph)

            # Take consecutive groups of five sentences
            for start in range(0, len(sentences) - 4, 5):

                group = sentences[start:start + 5]

                indexed = list(enumerate(group))

                # Shuffle the five sentences
                rng.shuffle(indexed)

                # Avoid accidentally generating the original order
                if [x[0] for x in indexed] == list(range(5)):
                    indexed.reverse()

                original_to_shuffled = {
                    original_index: shuffled_position + 1
                    for shuffled_position, (original_index, _)
                    in enumerate(indexed)
                }

                input_text = "\n".join(
                    f"S{i + 1}: {sentence}"
                    for i, (_, sentence) in enumerate(indexed)
                )

                target_text = " ".join(
                    f"S{original_to_shuffled[i]}"
                    for i in range(5)
                )

                yield {
                    "input_text": input_text,
                    "target_text": target_text
                }

In [20]:
ordering_features = Features({
    "input_text": Value("string"),
    "target_text": Value("string")
})


def build_ordering_dataset(source_dataset, split_name):

    return Dataset.from_generator(
        generate_ordering_examples,
        gen_kwargs={
            "source_dataset": source_dataset
        },
        features=ordering_features,
        split=split_name,
        keep_in_memory=False,
        writer_batch_size=500
    )


order_train = build_ordering_dataset(
    wiki_splits["train"], "train"
)

order_val = build_ordering_dataset(
    wiki_splits["validation"], "validation"
)

order_test = build_ordering_dataset(
    wiki_splits["test"], "test"
)


print("Training examples:", len(order_train))
print("Validation examples:", len(order_val))
print("Testing examples:", len(order_test))

print("\nSample input:")
print(order_train[0]["input_text"])

print("\nExpected order:")
print(order_train[0]["target_text"])

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Training examples: 121408
Validation examples: 14845
Testing examples: 15431

Sample input:
S1: प्रारंभ में (१९९५-२००७ तक) प्रवेश एम.जे.पी.
S2: यह एम॰जे॰पी॰ रोहिलखंड विश्वविद्यालय के परिसर में स्थित है।
S3: यह बिहार प्रान्त में स्थित प्रतिष्ठित अभियांत्रिकी संस्थानों में से एक है जिसमें विभिन्न शाखाएं है।
S4: रोहिलखंड विश्वविद्यालय के स्वयं के अभियांत्रिकी प्रवेश परीक्षा के माध्यम से अखिल भारतीय स्तर पर आयोजित किया गया था।
S5: इंजीनियरिंग और प्रौद्योगिकी संस्थान, बरेली (आईईटी) उत्तर प्रदेश, बरेली में राज्य सरकार द्वारा वित्त पोषित इंजीनियरिंग कॉलेज है।

Expected order:
S5 S2 S3 S1 S4


In [21]:
translation_raw = load_dataset(
    "cfilt/iitb-english-hindi",
    split="train"
)

'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/cfilt/iitb-english-hindi/resolve/321516f50bdcc1214fa75164c545478976ed84bd/iitb-english-hindi.py
Retrying in 1s [Retry 1/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/cfilt/iitb-english-hindi/resolve/321516f50bdcc1214fa75164c545478976ed84bd/iitb-english-hindi.py
Retrying in 2s [Retry 2/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/cfilt/iitb-english-hindi/resolve/321516f50bdcc1214fa75164c545478976ed84bd/iitb-english-hindi.py
Retrying in 4s [Retry 3/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/cfilt/iitb-english-hindi/resolve/321516f50bdcc1214fa75164c545478976ed84bd/iitb-english-hindi.py
Retrying in 8s [Retry 4/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/cfilt/iitb-english-hindi

In [22]:
def extract_translation(batch):

    return {
        "source_text": [
            clean_hindi(item["hi"])
            for item in batch["translation"]
        ],

        "target_text": [
            clean_hindi(item["en"])
            for item in batch["translation"]
        ]
    }


translation_data = translation_raw.map(
    extract_translation,
    batched=True,
    batch_size=500,
    writer_batch_size=500,
    remove_columns=translation_raw.column_names,
    keep_in_memory=False
)

translation_data = translation_data.filter(
    lambda x: bool(x["source_text"]) and bool(x["target_text"])
)

print(translation_data)

print("\nHindi:", translation_data[0]["source_text"])
print("English:", translation_data[0]["target_text"])

Map:   0%|          | 0/1659083 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1659083 [00:00<?, ? examples/s]

Dataset({
    features: ['source_text', 'target_text'],
    num_rows: 1650437
})

Hindi: अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें
English: Give your application an accessibility workout


In [23]:
translation_split = translation_data.train_test_split(
    test_size=0.20,
    seed=42
)

translation_val_test = translation_split["test"].train_test_split(
    test_size=0.50,
    seed=42
)

translation_data = DatasetDict({
    "train": translation_split["train"],
    "validation": translation_val_test["train"],
    "test": translation_val_test["test"]
})

print(translation_data)

DatasetDict({
    train: Dataset({
        features: ['source_text', 'target_text'],
        num_rows: 1320349
    })
    validation: Dataset({
        features: ['source_text', 'target_text'],
        num_rows: 165044
    })
    test: Dataset({
        features: ['source_text', 'target_text'],
        num_rows: 165044
    })
})


In [24]:
mt5_tokenizer = AutoTokenizer.from_pretrained(
    "google/mt5-small"
)

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

d:\punctuation_project\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HOME\.cache\huggingface\hub\models--google--mt5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

In [25]:
def tokenize_ordering(batch):

    inputs = [
        "restore order: " + text
        for text in batch["input_text"]
    ]

    model_inputs = mt5_tokenizer(
        inputs,
        max_length=512,
        truncation=True
    )

    targets = mt5_tokenizer(
        text_target=batch["target_text"],
        max_length=32,
        truncation=True
    )

    model_inputs["labels"] = targets["input_ids"]

    return model_inputs


ordering_data = DatasetDict({
    "train": order_train.map(
        tokenize_ordering,
        batched=True,
        batch_size=32,
        remove_columns=order_train.column_names,
        keep_in_memory=False
    ),

    "validation": order_val.map(
        tokenize_ordering,
        batched=True,
        batch_size=32,
        remove_columns=order_val.column_names,
        keep_in_memory=False
    ),

    "test": order_test.map(
        tokenize_ordering,
        batched=True,
        batch_size=32,
        remove_columns=order_test.column_names,
        keep_in_memory=False
    )
})

print(ordering_data)

Map:   0%|          | 0/121408 [00:00<?, ? examples/s]

Map:   0%|          | 0/14845 [00:00<?, ? examples/s]

Map:   0%|          | 0/15431 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 121408
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 14845
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 15431
    })
})


In [26]:

PROJECT_DIR = Path.cwd()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

ordering_save_path = PROCESSED_DIR / "ordering_processed"

ordering_data.save_to_disk(str(ordering_save_path))

print("Dataset saved at:", ordering_save_path)

Saving the dataset (0/1 shards):   0%|          | 0/121408 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/14845 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/15431 [00:00<?, ? examples/s]

Dataset saved at: d:\punctuation_project\data\processed\ordering_processed


In [27]:
def tokenize_translation(batch):

    inputs = [
        "translate Hindi to English: " + text
        for text in batch["source_text"]
    ]

    model_inputs = mt5_tokenizer(
        inputs,
        max_length=128,
        truncation=True
    )

    targets = mt5_tokenizer(
        text_target=batch["target_text"],
        max_length=128,
        truncation=True
    )

    model_inputs["labels"] = targets["input_ids"]

    return model_inputs


translation_tokenized = translation_data.map(
    tokenize_translation,
    batched=True,
    batch_size=32,
    remove_columns=translation_data["train"].column_names,
    keep_in_memory=False
)

print(translation_tokenized)

Map:   0%|          | 0/1320349 [00:00<?, ? examples/s]

Map:   0%|          | 0/165044 [00:00<?, ? examples/s]

Map:   0%|          | 0/165044 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1320349
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 165044
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 165044
    })
})


In [28]:
PROJECT_DIR = Path.cwd()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

translation_save_path = PROCESSED_DIR / "translation_processed"

translation_tokenized.save_to_disk(str(translation_save_path))

print("Dataset saved at:", translation_save_path)

Saving the dataset (0/1 shards):   0%|          | 0/1320349 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/165044 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/165044 [00:00<?, ? examples/s]

Dataset saved at: d:\punctuation_project\data\processed\translation_processed


In [29]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / (1024**3),
            2
        ),
        "GB"
    )

print("\nPunctuation datasets:")
print(punctuation_data)

print("\nOrdering datasets:")
print(ordering_data)

print("\nTranslation datasets:")
print(translation_tokenized)

GPU available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
GPU memory: 4.0 GB

Punctuation datasets:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2251236
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 278086
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 283654
    })
})

Ordering datasets:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 121408
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 14845
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 15431
    })
})

Translation datasets:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1320349
    })
 

In [30]:
import torch

MAX_GPU_MEMORY_GB = 1.5

def check_gpu_memory():
    if not torch.cuda.is_available():
        print("GPU unavailable")
        return

    allocated = torch.cuda.memory_allocated() / (1024 ** 3)
    reserved = torch.cuda.memory_reserved() / (1024 ** 3)
    peak = torch.cuda.max_memory_allocated() / (1024 ** 3)

    print(f"Allocated: {allocated:.3f} GB")
    print(f"Reserved:  {reserved:.3f} GB")
    print(f"Peak:      {peak:.3f} GB")

    if max(allocated, reserved, peak) >= MAX_GPU_MEMORY_GB:
        raise RuntimeError("GPU memory limit reached!")

In [6]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

if not torch.cuda.is_available():
    raise RuntimeError("GPU is not available.")

MAX_GPU_MEMORY_GB = 1.5

total_gb = (
    torch.cuda.get_device_properties(0).total_memory
    / (1024 ** 3)
)

PYTORCH_MEMORY_LIMIT_GB = 1.0

memory_fraction = PYTORCH_MEMORY_LIMIT_GB / total_gb

torch.cuda.set_per_process_memory_fraction(
    memory_fraction,
    device=0
)

print("GPU:", torch.cuda.get_device_name(0))
print(f"Total GPU memory: {total_gb:.2f} GB")
print(
    f"PyTorch allocation limit: "
    f"{PYTORCH_MEMORY_LIMIT_GB:.2f} GB"
)

GPU: NVIDIA GeForce RTX 3050 Laptop GPU
Total GPU memory: 4.00 GB
PyTorch allocation limit: 1.00 GB


In [8]:
from transformers import AutoModelForTokenClassification

MODEL_NAME = "ai4bharat/indic-bert"

# Confirm that the processed dataset has the expected columns.
print(punctuation_data["train"].column_names)
print(punctuation_data["train"][0])

# Check GPU availability.
if not torch.cuda.is_available():
    raise RuntimeError("GPU is not available. Stop here.")

# Load IndicBERT with five output classes.
punctuation_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5,
    ignore_mismatched_sizes=True
)

# Move the model to GPU.
punctuation_model = punctuation_model.to("cuda")

print("IndicBERT loaded successfully.")

print(
    "Model parameters:",
    sum(p.numel() for p in punctuation_model.parameters())
)

print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / (1024 ** 3), 3),
    "GB"
)

['input_ids', 'attention_mask', 'labels']
{'input_ids': [2, 20252, 367, 15267, 44395, 13577, 28689, 670, 1143, 40006, 121176, 1302, 23269, 1301, 13577, 28689, 670, 5650, 11611, 1883, 163202, 155951, 5458, 85574, 25716, 1134, 9327, 1134, 2092, 1301, 45, 43832, 37025, 4157, 74174, 49706, 670, 1883, 25716, 1134, 142899, 45665, 4384, 4765, 77248, 2092, 14229, 9017, 13577, 28689, 670, 5650, 11611, 2092, 37, 103871, 229, 4384, 5320, 3], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, -100, 0, -100, 0, -100, -100, 1, -100, -100, -100, 0, 0, 0, -100, -100, 0, -100, 0, 0, -100, 0, -100, 0, -100, -100, -100, 0, 0, 0, 0, -100, 0, -100, 0, -100, 0, 0, -100, 0, -100, 0, 1, 0, 0, 0, 0, 0, -100, -100, 0, -100, 0, 0, -100, -100, 0, -100, 2, -100]}


Loading weights:   0%|          | 0/23 [00:00<?, ?it/s]

[transformers] AlbertForTokenClassification LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     | 
---------------------------------+------------+-
predictions.dense.weight         | UNEXPECTED | 
albert.pooler.bias               | UNEXPECTED | 
predictions.bias                 | UNEXPECTED | 
albert.pooler.weight             | UNEXPECTED | 
predictions.LayerNorm.weight     | UNEXPECTED | 
sop_classifier.classifier.bias   | UNEXPECTED | 
predictions.dense.bias           | UNEXPECTED | 
predictions.decoder.bias         | UNEXPECTED | 
sop_classifier.classifier.weight | UNEXPECTED | 
predictions.decoder.weight       | UNEXPECTED | 
predictions.LayerNorm.bias       | UNEXPECTED | 
classifier.bias                  | MISSING    | 
classifier.weight                | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missin

IndicBERT loaded successfully.
Model parameters: 32856837
GPU memory allocated: 0.123 GB


In [9]:
train_small = punctuation_data["train"].select(range(5000))
validation_small = punctuation_data["validation"].select(range(500))

data_collator = DataCollatorForTokenClassification(
    tokenizer=indic_bert_tokenizer,
    padding=True
)

In [34]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    predictions = np.argmax(predictions, axis=-1)

    # Ignore padding and special tokens.
    mask = labels != -100

    correct = (predictions == labels) & mask
    accuracy = correct.sum() / mask.sum()

    return {"accuracy": float(accuracy)}

In [10]:
import torch

gc.collect()
torch.cuda.empty_cache()

print(
    "PyTorch allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 3),
    "GB"
)

print(
    "PyTorch reserved:",
    round(torch.cuda.memory_reserved() / 1024**3, 3),
    "GB"
)

PyTorch allocated: 0.123 GB
PyTorch reserved: 0.137 GB


In [ ]:
import inspect
from transformers import TrainingArguments

training_kwargs = {
    "output_dir": str(MODEL_DIR / "indicbert_punctuation"),
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "fp16": True,
    "gradient_checkpointing": True,
    "learning_rate": 2e-5,
    "num_train_epochs": 1,
    "max_steps": 100,
    "warmup_steps": 10,
    "logging_steps": 10,
    "eval_steps": 50,
    "save_strategy": "no",
    "dataloader_num_workers": 0,
    "dataloader_pin_memory": False,
    "report_to": "none",
    "load_best_model_at_end": False
}

parameters = inspect.signature(
    TrainingArguments.__init__
).parameters

if "eval_strategy" in parameters:
    training_kwargs["eval_strategy"] = "steps"
else:
    training_kwargs["evaluation_strategy"] = "steps"

training_args = TrainingArguments(**training_kwargs)

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 4.00 GiB of which 2.92 GiB is free. Process 28268 has 17179869184.00 GiB memory in use. Process 29808 has 17179869184.00 GiB memory in use. Process 8484 has 17179869184.00 GiB memory in use. Process 23548 has 17179869184.00 GiB memory in use. Process 13188 has 17179869184.00 GiB memory in use. Process 17608 has 17179869184.00 GiB memory in use. Process 7528 has 17179869184.00 GiB memory in use. Process 7036 has 17179869184.00 GiB memory in use. Process 18980 has 17179869184.00 GiB memory in use. Process 20684 has 17179869184.00 GiB memory in use. Process 11556 has 17179869184.00 GiB memory in use. Process 13588 has 17179869184.00 GiB memory in use. Process 14884 has 17179869184.00 GiB memory in use. Process 28956 has 17179869184.00 GiB memory in use. Process 14168 has 17179869184.00 GiB memory in use. Process 15080 has 17179869184.00 GiB memory in use. Process 28616 has 17179869184.00 GiB m

In [ ]:
save_path = "/content/indicbert_punctuation_final"

trainer.save_model(save_path)
indic_bert_tokenizer.save_pretrained(save_path)

print("Model saved at:", save_path)

In [15]:
from collections import Counter
import torch

def count_labels(dataset):
    counts = Counter()

    for example in dataset:
        counts.update(
            label for label in example["labels"]
            if label != -100
        )

    return counts

# Count labels in the training subset.
train_counts = count_labels(train_small)

# Calculate weights for all five classes.
counts = torch.tensor(
    [train_counts[i] for i in range(5)],
    dtype=torch.float32
)

if torch.any(counts == 0):
    raise ValueError("At least one class is missing from train_small.")

class_weights = torch.sqrt(counts.sum() / counts)
class_weights = class_weights / class_weights.mean()

print("Training label counts:", train_counts)
print("Class weights:", class_weights)

Training label counts: Counter({0: 71085, 2: 3655, 1: 3255, 3: 22, 4: 18})
Class weights: tensor([0.0385, 0.1801, 0.1699, 2.1902, 2.4213])


In [13]:
print("Weighted trainer exists:", "weighted_trainer" in globals())
print("Weighted training model exists:", "punctuation_model" in globals())

Weighted trainer exists: False
Weighted training model exists: True


In [16]:
print("train_small exists:", "train_small" in globals())
print("data_collator exists:", "data_collator" in globals())
print("class_weights exists:", "class_weights" in globals())
print("validation_small exists:", "validation_small" in globals())

train_small exists: True
data_collator exists: True
class_weights exists: True
validation_small exists: True


In [17]:
print("class_weights exists:", "class_weights" in globals())
print("Class weights:", class_weights)

class_weights exists: True
Class weights: tensor([0.0385, 0.1801, 0.1699, 2.1902, 2.4213])


In [18]:
import torch
import torch.nn.functional as F
from transformers import Trainer, TrainingArguments

class WeightedLossTrainer(Trainer):

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None
    ):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.logits

        loss = F.cross_entropy(
            logits.reshape(-1, 5),
            labels.reshape(-1),
            weight=class_weights.to(logits.device),
            ignore_index=-100
        )

        return (loss, outputs) if return_outputs else loss


weighted_args = TrainingArguments(
    output_dir=str(MODEL_DIR / "indicbert_punctuation_smoke"),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    fp16=True,
    learning_rate=1e-5,
    max_steps=100,
    logging_steps=10,
    save_strategy="no",
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    report_to="none"
)

weighted_trainer = WeightedLossTrainer(
    model=punctuation_model,
    args=weighted_args,
    train_dataset=train_small,
    data_collator=data_collator
)

torch.cuda.reset_peak_memory_stats()

weighted_result = weighted_trainer.train()

print("\nSMOKE TEST COMPLETE")
print("Training loss:", weighted_result.training_loss)
print(
    "Peak GPU memory allocated:",
    round(torch.cuda.max_memory_allocated() / (1024 ** 3), 3),
    "GB"
)

Step,Training Loss
10,12.048288
20,9.536553
30,8.046428
40,7.098422
50,6.036389
60,5.244811
70,5.347600
80,5.166420
90,4.887519
100,5.098339



SMOKE TEST COMPLETE
Training loss: 6.8510770416259765
Peak GPU memory allocated: 0.603 GB


In [19]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

prediction_output = weighted_trainer.predict(validation_small)

predictions = np.argmax(prediction_output.predictions, axis=-1)
actual_labels = prediction_output.label_ids

mask = actual_labels != -100

y_true = actual_labels[mask]
y_pred = predictions[mask]

print("\nCLASSIFICATION REPORT")

print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3, 4],
        target_names=[
            "No punctuation",
            "Comma-like",
            "Sentence ending",
            "Question mark",
            "Exclamation mark"
        ],
        zero_division=0,
        digits=4
    )
)

print("\nCONFUSION MATRIX")

print(
    confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3, 4]
    )
)


CLASSIFICATION REPORT
                  precision    recall  f1-score   support

  No punctuation     0.9607    0.9957    0.9779      6529
      Comma-like     0.0000    0.0000    0.0000       239
 Sentence ending     0.9058    0.9160    0.9109       357
   Question mark     0.0000    0.0000    0.0000         1
Exclamation mark     0.0000    0.0000    0.0000         2

        accuracy                         0.9579      7128
       macro avg     0.3733    0.3823    0.3778      7128
    weighted avg     0.9253    0.9579    0.9413      7128


CONFUSION MATRIX
[[6501    0   28    0    0]
 [ 235    0    4    0    0]
 [  30    0  327    0    0]
 [   0    0    1    0    0]
 [   1    0    1    0    0]]


In [20]:

from collections import Counter
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

LABEL_NAMES = {
    0: "No punctuation",
    1: "Comma-like",
    2: "Sentence ending",
    3: "Question mark",
    4: "Exclamation mark"
}

# 1. Count actual labels in the validation dataset.
label_counts = Counter()

for example in validation_small:
    label_counts.update(
        label for label in example["labels"] if label != -100
    )

print("VALIDATION LABEL DISTRIBUTION")
for label, name in LABEL_NAMES.items():
    print(f"{name:20s}: {label_counts[label]:,}")

# 2. Get model predictions using the weighted trainer.
prediction_output = weighted_trainer.predict(validation_small)

predictions = np.argmax(prediction_output.predictions, axis=-1)
actual_labels = prediction_output.label_ids

# 3. Ignore special tokens and subword positions (-100).
mask = actual_labels != -100

y_true = actual_labels[mask]
y_pred = predictions[mask]

# 4. Evaluate each punctuation class separately.
print("\nCLASSIFICATION REPORT")
print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3, 4],
        target_names=[LABEL_NAMES[i] for i in range(5)],
        zero_division=0,
        digits=4
    )
)

print("\nCONFUSION MATRIX")
print(
    confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3, 4]
    )
)

VALIDATION LABEL DISTRIBUTION
No punctuation      : 6,529
Comma-like          : 239
Sentence ending     : 357
Question mark       : 1
Exclamation mark    : 2



CLASSIFICATION REPORT
                  precision    recall  f1-score   support

  No punctuation     0.9607    0.9957    0.9779      6529
      Comma-like     0.0000    0.0000    0.0000       239
 Sentence ending     0.9058    0.9160    0.9109       357
   Question mark     0.0000    0.0000    0.0000         1
Exclamation mark     0.0000    0.0000    0.0000         2

        accuracy                         0.9579      7128
       macro avg     0.3733    0.3823    0.3778      7128
    weighted avg     0.9253    0.9579    0.9413      7128


CONFUSION MATRIX
[[6501    0   28    0    0]
 [ 235    0    4    0    0]
 [  30    0  327    0    0]
 [   0    0    1    0    0]
 [   1    0    1    0    0]]


In [21]:
from collections import Counter

LABEL_NAMES = {
    0: "No punctuation",
    1: "Comma-like",
    2: "Sentence ending",
    3: "Question mark",
    4: "Exclamation mark"
}

def count_labels(dataset):
    counts = Counter()

    for example in dataset:
        counts.update(
            label
            for label in example["labels"]
            if label != -100
        )

    return counts


train_counts = count_labels(punctuation_data["train"])
val_counts = count_labels(punctuation_data["validation"])

print("FULL TRAINING DATA")
for label, name in LABEL_NAMES.items():
    print(f"{name:20s}: {train_counts[label]:,}")

print("\nFULL VALIDATION DATA")
for label, name in LABEL_NAMES.items():
    print(f"{name:20s}: {val_counts[label]:,}")

FULL TRAINING DATA
No punctuation      : 32,328,547
Comma-like          : 1,469,218
Sentence ending     : 1,641,962
Question mark       : 6,268
Exclamation mark    : 5,472

FULL VALIDATION DATA
No punctuation      : 3,976,483
Comma-like          : 183,409
Sentence ending     : 202,158
Question mark       : 751
Exclamation mark    : 844


In [22]:
import torch
import numpy as np

# Use the label counts from the complete training dataset.
counts = torch.tensor(
    [
        train_counts[0],
        train_counts[1],
        train_counts[2],
        train_counts[3],
        train_counts[4]
    ],
    dtype=torch.float32
)

# Square-root inverse-frequency weighting.
class_weights = torch.sqrt(counts.sum() / counts)

# Normalize weights to have an average of 1.
class_weights = class_weights / class_weights.mean()

print("Class weights:")

for i, weight in enumerate(class_weights):
    print(f"{LABEL_NAMES[i]:20s}: {weight.item():.4f}")

Class weights:
No punctuation      : 0.0315
Comma-like          : 0.1477
Sentence ending     : 0.1397
Question mark       : 2.2611
Exclamation mark    : 2.4200


In [23]:
import torch.nn.functional as F
from transformers import Trainer, TrainingArguments
import inspect

class WeightedLossTrainer(Trainer):

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None
    ):
        labels = inputs.pop("labels")

        outputs = model(**inputs)

        logits = outputs.logits

        loss = F.cross_entropy(
            logits.reshape(-1, 5),
            labels.reshape(-1),
            weight=class_weights.to(logits.device),
            ignore_index=-100
        )

        return (loss, outputs) if return_outputs else loss


weighted_kwargs = {
    "output_dir": "/content/indicbert_punctuation_weighted",
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "fp16": True,
    "learning_rate": 1e-5,
    "max_steps": 2000,
    "logging_steps": 100,
    "save_strategy": "no",
    "dataloader_num_workers": 0,
    "dataloader_pin_memory": False,
    "report_to": "none"
}

weighted_args = TrainingArguments(**weighted_kwargs)

weighted_trainer = WeightedLossTrainer(
    model=punctuation_model,
    args=weighted_args,
    train_dataset=punctuation_data["train"],
    data_collator=data_collator
)

torch.cuda.reset_peak_memory_stats()

weighted_result = weighted_trainer.train()

print("Weighted training completed.")
print("Training loss:", weighted_result.training_loss)

print(
    "Peak PyTorch GPU memory:",
    round(torch.cuda.max_memory_allocated() / (1024 ** 3), 3),
    "GB"
)

!nvidia-smi

Step,Training Loss
100,3.962570
200,3.250965
300,2.954265
400,3.127263
500,2.832632
600,2.541033
700,2.731937
800,2.740503
900,2.592627
1000,2.459179


Weighted training completed.
Training loss: 2.718974060058594
Peak PyTorch GPU memory: 0.603 GB
Wed Nov 11 14:31:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 566.07                 Driver Version: 566.07         CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
| N/A   57C    P3             14W /   75W |    1714MiB /   4096MiB |     35%      Default |
|                                         | 

In [ ]:
validation_result = weighted_trainer.evaluate(
    eval_dataset=punctuation_data["validation"]
)

print(validation_result)

In [24]:
final_model_path = MODEL_DIR / "indicbert_punctuation_weighted_final"

weighted_trainer.save_model(str(final_model_path))

indic_bert_tokenizer.save_pretrained(str(final_model_path))

print("Model saved successfully.")
print("Saved location:", final_model_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully.
Saved location: d:\punctuation_project\models\indicbert_punctuation_weighted_final


In [25]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix

# Use the trained model.
model = weighted_trainer.model
model.eval()

# Process one example at a time to keep GPU memory low.
validation_loader = DataLoader(
    punctuation_data["validation"],
    batch_size=1,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=0,
    pin_memory=False
)

# Allocate the final predictions on CPU, not GPU.
total_labels = sum(val_counts.values())

all_true = np.empty(total_labels, dtype=np.uint8)
all_pred = np.empty(total_labels, dtype=np.uint8)

position = 0

torch.cuda.reset_peak_memory_stats()

with torch.inference_mode():

    for step, batch in enumerate(validation_loader):

        # Keep labels on CPU.
        labels = batch.pop("labels")

        # Move only model inputs to GPU.
        inputs = {
            key: value.to("cuda")
            for key, value in batch.items()
        }

        # Predict.
        outputs = model(**inputs)
        predictions = outputs.logits.argmax(dim=-1).cpu()

        # Remove padding and special tokens.
        mask = labels != -100

        true_batch = labels[mask].numpy().astype(np.uint8)
        pred_batch = predictions[mask].numpy().astype(np.uint8)

        # Store results in CPU arrays.
        batch_size = len(true_batch)

        all_true[position:position + batch_size] = true_batch
        all_pred[position:position + batch_size] = pred_batch

        position += batch_size

        # Release batch tensors.
        del outputs, predictions, inputs, labels, batch

        if (step + 1) % 5000 == 0:
            print(f"Processed {step + 1:,} validation examples")

print("\nEvaluation completed.")
print("Total evaluated tokens:", position)

print(
    "Peak PyTorch GPU memory:",
    round(torch.cuda.max_memory_allocated() / (1024 ** 3), 3),
    "GB"
)

print("Expected validation tokens:", total_labels)
print("Actually evaluated tokens:", position)

if position != total_labels:
    raise ValueError("Validation token count does not match.")

print("\nCLASSIFICATION REPORT")

print(
    classification_report(
        all_true[:position],
        all_pred[:position],
        labels=[0, 1, 2, 3, 4],
        target_names=[
            "No punctuation",
            "Comma-like",
            "Sentence ending",
            "Question mark",
            "Exclamation mark"
        ],
        zero_division=0,
        digits=4
    )
)

print("\nCONFUSION MATRIX")

print(
    confusion_matrix(
        all_true[:position],
        all_pred[:position],
        labels=[0, 1, 2, 3, 4]
    )
)

!nvidia-smi

Processed 5,000 validation examples
Processed 10,000 validation examples
Processed 15,000 validation examples
Processed 20,000 validation examples
Processed 25,000 validation examples
Processed 30,000 validation examples
Processed 35,000 validation examples
Processed 40,000 validation examples
Processed 45,000 validation examples
Processed 50,000 validation examples
Processed 55,000 validation examples
Processed 60,000 validation examples
Processed 65,000 validation examples
Processed 70,000 validation examples
Processed 75,000 validation examples
Processed 80,000 validation examples
Processed 85,000 validation examples
Processed 90,000 validation examples
Processed 95,000 validation examples
Processed 100,000 validation examples
Processed 105,000 validation examples
Processed 110,000 validation examples
Processed 115,000 validation examples
Processed 120,000 validation examples
Processed 125,000 validation examples
Processed 130,000 validation examples
Processed 135,000 validation exa

In [5]:

# Inspect the original Wikipedia article used for ordering data

example = dataset["train"][0]

print("CURRENT PROCESSED SENTENCES")
print("=" * 80)

for i, ids in enumerate(example["sentence_input_ids"]):
    mask = example["sentence_attention_mask"][i]

    valid_ids = [
        token_id
        for token_id, attention in zip(ids, mask)
        if attention == 1
    ]

    print(f"\nSentence {i+1}:")
    print(tokenizer.decode(valid_ids, skip_special_tokens=True))


print("\n\nORIGINAL WIKIPEDIA ARTICLE")
print("=" * 80)

# Search the original article dataset for a matching phrase.
# Use a distinctive word sequence from the decoded sentence.
search_phrase = "रहलखड"

matches = []

for article in wiki_splits["train"]:
    if search_phrase in article["text"]:
        matches.append(article["text"])
        if len(matches) == 3:
            break

print("Matching articles:", len(matches))

for i, article_text in enumerate(matches):
    print(f"\nARTICLE {i+1}:")
    print(article_text[:3000])


NameError: name 'dataset' is not defined

In [6]:

from pathlib import Path

PROJECT_DIR = Path(r"E:\punctuation_project")

print("Dataset-related folders:")
for path in PROJECT_DIR.rglob("*"):
    if path.is_dir() and any(
        word in path.name.lower()
        for word in ["wiki", "punctuation", "ordering"]
    ):
        print(path)


Dataset-related folders:
E:\punctuation_project\data\processed\ordering_indicbert_processed
E:\punctuation_project\data\processed\ordering_indicbert_sentence_processed
E:\punctuation_project\data\processed\ordering_processed
E:\punctuation_project\data\processed\punctuation_processed
E:\punctuation_project\models\indicbert_punctuation_smoke
E:\punctuation_project\models\indicbert_punctuation_weighted_final
E:\punctuation_project\models\sentence_ordering_indicbert
E:\punctuation_project\models\sentence_ordering_transformer


In [7]:

from datasets import load_from_disk
from pathlib import Path

PROJECT_DIR = Path(r"E:\punctuation_project")

# Load the existing sentence-level ordering dataset
DATASET_PATH = PROJECT_DIR / "data" / "processed" / "ordering_indicbert_sentence_processed"

dataset = load_from_disk(str(DATASET_PATH))

print("Dataset loaded successfully")
print("Splits:", list(dataset.keys()))

for split in dataset:
    print(f"{split}: {len(dataset[split])} examples")

example = dataset["train"][0]

print("\nFirst example labels:", example["labels"])


Dataset loaded successfully
Splits: ['train', 'validation', 'test']
train: 121316 examples
validation: 14835 examples
test: 15417 examples

First example labels: [4, 1, 2, 0, 3]


In [8]:

import json
from pathlib import Path

notebook_path = Path(
    r"E:\punctuation_project\Add-Punctuation-and-Translate.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

keywords = [
    "wiki_dataset",
    "wiki_splits",
    "save_to_disk",
    "load_dataset"
]

for i, cell in enumerate(notebook["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    if any(keyword in source for keyword in keywords):
        print(f"\n{'=' * 70}")
        print("CELL:", i)
        print("=" * 70)
        print(source[:5000])



CELL: 2
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModelForSeq2SeqLM, Trainer, TrainingArguments, Seq2SeqTrainer, Seq2SeqTrainingArguments
from datasets import load_dataset

CELL: 3
import os
import re
import gc
import random
import unicodedata
import regex
import numpy as np
import torch

from pathlib import Path
from collections import Counter

from datasets import (
    load_dataset,
    load_from_disk,
    Dataset,
    DatasetDict,
    Features,
    Sequence,
    Value
)

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import classification_report, confusion_matrix

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODEL_DIR = PROJECT_D

In [9]:

import re
import unicodedata
from datasets import load_dataset, DatasetDict

# 1. Reload the original Hindi Wikipedia dataset
print("Loading Hindi Wikipedia dataset...")

wiki_dataset = load_dataset(
    "wikimedia/wikipedia",
    "20231101.hi",
    split="train"
)

print("Total articles:", len(wiki_dataset))


# 2. Reproduce the EXACT original splits
wiki_temp = wiki_dataset.train_test_split(
    test_size=0.20,
    seed=42
)

wiki_remaining = wiki_temp["test"].train_test_split(
    test_size=0.50,
    seed=42
)

wiki_splits = DatasetDict({
    "train": wiki_temp["train"],
    "validation": wiki_remaining["train"],
    "test": wiki_remaining["test"]
})

print("Training articles:", len(wiki_splits["train"]))


# 3. Reproduce the original cleaning function
def clean_hindi(text):
    text = unicodedata.normalize("NFC", str(text))
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[\u200b\ufeff]", "", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


# 4. Reproduce the original sentence splitter
def split_sentences(text):
    sentences = re.split(
        r"(?<=[।॥!?])\s+|(?<=\.)\s+",
        text
    )

    return [
        sentence.strip()
        for sentence in sentences
        if len(sentence.strip().split()) >= 3
    ]


# 5. Find the first group of five sentences
found = False

for article_index, article in enumerate(wiki_splits["train"]):

    cleaned_article = clean_hindi(article["text"])

    for paragraph_index, paragraph in enumerate(
        cleaned_article.split("\n")
    ):

        sentences = split_sentences(paragraph)

        if len(sentences) >= 5:

            print("\n" + "=" * 80)
            print("FIRST ORIGINAL TRAINING GROUP")
            print("=" * 80)

            print("Article index:", article_index)
            print("Paragraph index:", paragraph_index)

            for i, sentence in enumerate(sentences[:5]):
                print(f"\nOriginal sentence {i+1}:")
                print(sentence)

            found = True
            break

    if found:
        break

if not found:
    print("No paragraph containing at least five sentences found.")


Loading Hindi Wikipedia dataset...
Total articles: 163093
Training articles: 130474

FIRST ORIGINAL TRAINING GROUP
Article index: 1
Paragraph index: 0

Original sentence 1:
इंजीनियरिंग और प्रौद्योगिकी संस्थान, बरेली (आईईटी) उत्तर प्रदेश, बरेली में राज्य सरकार द्वारा वित्त पोषित इंजीनियरिंग कॉलेज है।

Original sentence 2:
यह एम॰जे॰पी॰ रोहिलखंड विश्वविद्यालय के परिसर में स्थित है।

Original sentence 3:
यह बिहार प्रान्त में स्थित प्रतिष्ठित अभियांत्रिकी संस्थानों में से एक है जिसमें विभिन्न शाखाएं है।

Original sentence 4:
प्रारंभ में (१९९५-२००७ तक) प्रवेश एम.जे.पी.

Original sentence 5:
रोहिलखंड विश्वविद्यालय के स्वयं के अभियांत्रिकी प्रवेश परीक्षा के माध्यम से अखिल भारतीय स्तर पर आयोजित किया गया था।


In [10]:

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indic-bert")

original_sentences = [
    "इंजीनियरिंग और प्रौद्योगिकी संस्थान, बरेली (आईईटी) उत्तर प्रदेश, बरेली में राज्य सरकार द्वारा वित्त पोषित इंजीनियरिंग कॉलेज है।",
    "यह एम॰जे॰पी॰ रोहिलखंड विश्वविद्यालय के परिसर में स्थित है।",
    "प्रारंभ में (१९९५-२००७ तक) प्रवेश एम.जे.पी."
]

for sentence in original_sentences:

    encoded = tokenizer(
        sentence,
        add_special_tokens=False
    )

    decoded = tokenizer.decode(
        encoded["input_ids"],
        skip_special_tokens=True
    )

    print("\nORIGINAL:")
    print(sentence)

    print("\nDECODED:")
    print(decoded)

    print("\nTOKEN IDs:")
    print(encoded["input_ids"])

    print("-" * 80)




ORIGINAL:
इंजीनियरिंग और प्रौद्योगिकी संस्थान, बरेली (आईईटी) उत्तर प्रदेश, बरेली में राज्य सरकार द्वारा वित्त पोषित इंजीनियरिंग कॉलेज है।

DECODED:
इजनयरग और परदयगक ससथन, बरल (आईईट) उततर परदश, बरल म रजय सरकर दवर वतत पषत इजनयरग कलज ह।

TOKEN IDs:
[43885, 339, 10495, 2389, 31, 37, 2525, 1134, 2389, 507, 152133, 96276, 12, 9017, 487, 20, 14390, 162898, 129, 118104, 9327, 37, 156429, 12, 9017, 487, 2092, 34886, 1134, 3841, 508, 2155, 555, 130, 229, 229, 2543, 11611, 229, 43885, 339, 10495, 2389, 2899, 1325, 4384, 15]
--------------------------------------------------------------------------------

ORIGINAL:
यह एम॰जे॰पी॰ रोहिलखंड विश्वविद्यालय के परिसर में स्थित है।

DECODED:
यह एम॰ज॰प॰ रहलखड वशववदयलय क परसर म सथत ह।

TOKEN IDs:
[114, 3878, 64247, 1325, 64247, 1551, 64247, 2022, 487, 89201, 77636, 1976, 109832, 28689, 1134, 1883, 37, 6511, 2092, 1301, 4090, 229, 4384, 15]
--------------------------------------------------------------------------------

ORIGINAL:
प्रारंभ में (१९९५-२००७ तक) 